# 04 Algorithm Comparison - K-Means vs Hierarchical vs DBSCAN

This notebook demonstrates:
- Comparing clustering results from different algorithms
- Analyzing algorithm agreement/disagreement
- Understanding algorithm-specific strengths
- Robustness analysis

## Setup

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Algorithm Results

In [ ]:
# Load results from each algorithm
base_path = Path('../output/germany/02_algorithms')

# K-Means
kmeans_path = base_path / 'kmeans_comparative/combined/data/combined_data.csv'
# Hierarchical
hierarchical_path = base_path / 'hierarchical/hierarchical_results.csv'
# DBSCAN
dbscan_path = base_path / 'dbscan/dbscan_results.csv'

results = {}

# Try loading each algorithm
if kmeans_path.exists():
    results['kmeans'] = pd.read_csv(kmeans_path)
    print(f"✓ K-Means: {len(results['kmeans'])} companies, {results['kmeans']['cluster'].nunique()} clusters")

if hierarchical_path.exists():
    results['hierarchical'] = pd.read_csv(hierarchical_path)
    print(f"✓ Hierarchical: {len(results['hierarchical'])} companies, {results['hierarchical']['cluster'].nunique()} clusters")

if dbscan_path.exists():
    results['dbscan'] = pd.read_csv(dbscan_path)
    n_clusters = len([c for c in results['dbscan']['cluster'].unique() if c != -1])
    n_noise = (results['dbscan']['cluster'] == -1).sum()
    print(f"✓ DBSCAN: {len(results['dbscan'])} companies, {n_clusters} clusters, {n_noise} noise points")

if not results:
    print("⚠ No algorithm results found - run pipeline first")
else:
    print(f"\n📊 Loaded results from {len(results)} algorithm(s)")

## 2. Cluster Size Comparison

In [ ]:
if len(results) >= 2:
    fig, axes = plt.subplots(1, len(results), figsize=(6*len(results), 5))
    if len(results) == 1:
        axes = [axes]
    
    for i, (algo_name, df) in enumerate(results.items()):
        ax = axes[i]
        
        # Get cluster sizes (exclude noise for DBSCAN)
        if algo_name == 'dbscan':
            cluster_sizes = df[df['cluster'] != -1]['cluster'].value_counts().sort_index()
        else:
            cluster_sizes = df['cluster'].value_counts().sort_index()
        
        cluster_sizes.plot(kind='bar', ax=ax, color='steelblue')
        ax.set_title(f'{algo_name.upper()}\n{len(cluster_sizes)} clusters', 
                     fontsize=12, fontweight='bold')
        ax.set_xlabel('Cluster ID')
        ax.set_ylabel('Number of Companies')
        ax.tick_params(axis='x', rotation=0)
    
    plt.suptitle('Cluster Size Comparison Across Algorithms', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Algorithm Agreement Analysis

Measure how much algorithms agree on clustering assignments.

In [ ]:
if len(results) >= 2:
    # Merge results on gvkey
    algo_names = list(results.keys())
    
    # Start with first algorithm
    merged = results[algo_names[0]][['gvkey', 'cluster']].rename(
        columns={'cluster': f'cluster_{algo_names[0]}'}
    )
    
    # Merge others
    for algo_name in algo_names[1:]:
        merged = merged.merge(
            results[algo_name][['gvkey', 'cluster']].rename(
                columns={'cluster': f'cluster_{algo_name}'}
            ),
            on='gvkey',
            how='inner'
        )
    
    print(f"✓ Merged {len(merged)} companies across {len(algo_names)} algorithms")
    
    # Calculate pairwise agreement metrics
    print("\n📊 Pairwise Agreement Metrics:\n")
    
    agreement_matrix = pd.DataFrame(index=algo_names, columns=algo_names)
    nmi_matrix = pd.DataFrame(index=algo_names, columns=algo_names)
    
    for algo1 in algo_names:
        for algo2 in algo_names:
            if algo1 == algo2:
                agreement_matrix.loc[algo1, algo2] = 1.0
                nmi_matrix.loc[algo1, algo2] = 1.0
            else:
                labels1 = merged[f'cluster_{algo1}']
                labels2 = merged[f'cluster_{algo2}']
                
                # Adjusted Rand Index
                ari = adjusted_rand_score(labels1, labels2)
                agreement_matrix.loc[algo1, algo2] = ari
                
                # Normalized Mutual Information
                nmi = normalized_mutual_info_score(labels1, labels2)
                nmi_matrix.loc[algo1, algo2] = nmi
    
    # Convert to numeric
    agreement_matrix = agreement_matrix.astype(float)
    nmi_matrix = nmi_matrix.astype(float)
    
    print("Adjusted Rand Index (ARI):")
    print(agreement_matrix.round(3))
    print("\nNormalized Mutual Information (NMI):")
    print(nmi_matrix.round(3))
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.heatmap(agreement_matrix, annot=True, fmt='.3f', cmap='RdYlGn', 
                center=0.5, vmin=0, vmax=1, ax=ax1, cbar_kws={'label': 'ARI Score'})
    ax1.set_title('Adjusted Rand Index', fontweight='bold')
    
    sns.heatmap(nmi_matrix, annot=True, fmt='.3f', cmap='Blues', 
                vmin=0, vmax=1, ax=ax2, cbar_kws={'label': 'NMI Score'})
    ax2.set_title('Normalized Mutual Information', fontweight='bold')
    
    plt.suptitle('Algorithm Agreement Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Interpretation:")
    print("  • ARI = 1: Perfect agreement")
    print("  • ARI = 0: Random agreement")
    print("  • ARI < 0: Worse than random")
    print("  • NMI ∈ [0,1]: Higher = more agreement")

## 4. Companies with Different Assignments

Identify companies that are clustered differently by algorithms.

In [ ]:
if len(results) >= 2 and 'merged' in locals():
    # Find companies with different assignments
    algo_cols = [f'cluster_{algo}' for algo in algo_names]
    
    # Check if all algorithms agree
    merged['all_agree'] = merged[algo_cols].nunique(axis=1) == 1
    
    agreement_rate = merged['all_agree'].sum() / len(merged) * 100
    
    print(f"📊 Agreement Statistics:")
    print(f"   • {merged['all_agree'].sum()} companies ({agreement_rate:.1f}%) - All algorithms agree")
    print(f"   • {(~merged['all_agree']).sum()} companies ({100-agreement_rate:.1f}%) - Algorithms disagree")
    
    # Sample of disagreements
    disagreements = merged[~merged['all_agree']]
    
    if len(disagreements) > 0:
        print(f"\n🔍 Sample Disagreements (first 10):")
        
        # Add company names if available
        if 'company_name' in results[algo_names[0]].columns:
            disagreements = disagreements.merge(
                results[algo_names[0]][['gvkey', 'company_name']],
                on='gvkey',
                how='left'
            )
            print(disagreements[['gvkey', 'company_name'] + algo_cols].head(10))
        else:
            print(disagreements[['gvkey'] + algo_cols].head(10))

## 5. Contingency Table Analysis

Cross-tabulation showing how clusters map between algorithms.

In [ ]:
if len(results) >= 2 and 'merged' in locals():
    # Create contingency table for first two algorithms
    algo1, algo2 = algo_names[0], algo_names[1]
    
    contingency = pd.crosstab(
        merged[f'cluster_{algo1}'],
        merged[f'cluster_{algo2}'],
        margins=True
    )
    
    print(f"📊 Contingency Table: {algo1.upper()} vs {algo2.upper()}")
    print(f"\nRows = {algo1.upper()} clusters, Columns = {algo2.upper()} clusters\n")
    print(contingency)
    
    # Visualize (without margins)
    contingency_no_margins = contingency.iloc[:-1, :-1]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(contingency_no_margins, annot=True, fmt='d', cmap='Blues', 
                cbar_kws={'label': 'Number of Companies'})
    plt.title(f'Cluster Mapping: {algo1.upper()} vs {algo2.upper()}', 
              fontsize=14, fontweight='bold')
    plt.xlabel(f'{algo2.upper()} Cluster')
    plt.ylabel(f'{algo1.upper()} Cluster')
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Interpretation:")
    print("  • High values on diagonal = Good agreement")
    print("  • Scattered values = Algorithms cluster differently")

## 6. Algorithm-Specific Insights

In [ ]:
print("🔍 Algorithm Characteristics:\n")

if 'kmeans' in results:
    print("K-MEANS:")
    print("  ✓ Strengths: Fast, scalable, works well with spherical clusters")
    print("  ⚠ Limitations: Requires pre-set k, sensitive to outliers")
    print(f"  • Found {results['kmeans']['cluster'].nunique()} clusters")
    print()

if 'hierarchical' in results:
    print("HIERARCHICAL:")
    print("  ✓ Strengths: No need to pre-set k, creates dendrogram")
    print("  ⚠ Limitations: Slower, sensitive to linkage method")
    print(f"  • Found {results['hierarchical']['cluster'].nunique()} clusters")
    print()

if 'dbscan' in results:
    n_clusters = len([c for c in results['dbscan']['cluster'].unique() if c != -1])
    n_noise = (results['dbscan']['cluster'] == -1).sum()
    print("DBSCAN:")
    print("  ✓ Strengths: Finds arbitrary shapes, identifies outliers")
    print("  ⚠ Limitations: Sensitive to eps/min_samples, struggles with varying densities")
    print(f"  • Found {n_clusters} clusters + {n_noise} noise points")
    print()

## 7. Robustness Summary

Overall assessment of clustering robustness.

In [ ]:
if len(results) >= 2 and 'agreement_matrix' in locals():
    # Calculate average agreement (excluding diagonal)
    mask = ~np.eye(len(agreement_matrix), dtype=bool)
    avg_ari = agreement_matrix.values[mask].mean()
    avg_nmi = nmi_matrix.values[mask].mean()
    
    print("📊 Clustering Robustness Summary:\n")
    print(f"Average ARI across algorithms: {avg_ari:.3f}")
    print(f"Average NMI across algorithms: {avg_nmi:.3f}")
    print(f"\nCompanies with unanimous agreement: {agreement_rate:.1f}%")
    
    print("\n💡 Robustness Assessment:")
    if avg_ari > 0.7:
        print("  ✓ STRONG: Algorithms show high agreement")
    elif avg_ari > 0.4:
        print("  ~ MODERATE: Algorithms partially agree")
    else:
        print("  ⚠ WEAK: Algorithms show low agreement")
    
    print("\n📝 Recommendations:")
    if avg_ari > 0.6:
        print("  • Clustering is robust - safe to use for thesis")
        print("  • Focus on K-Means for main analysis (most interpretable)")
        print("  • Use disagreements for robustness discussion")
    else:
        print("  • Consider adjusting parameters (k, eps, linkage)")
        print("  • Investigate why algorithms disagree")
        print("  • May need different preprocessing or feature selection")

## Next Steps

- **01_quick_start.ipynb**: Quick data exploration
- **02_config_tuning.ipynb**: Adjust parameters to improve agreement
- **03_cluster_exploration.ipynb**: Deep dive into specific clusters
- **99_full_pipeline.ipynb**: Run complete analysis with optimal parameters

## For Thesis

Use this analysis to:
1. **Justify algorithm choice**: Show why K-Means/Hierarchical is preferred
2. **Demonstrate robustness**: Report agreement metrics in methodology
3. **Discuss limitations**: Use disagreements to show critical thinking
4. **Triangulate findings**: Validate main findings across algorithms